# Coluna tutorial: column generation

From: https://atoptima.github.io/Coluna.jl/latest/start/start/

In [1]:
using Pkg
Pkg.activate("../")

  Activating project at `c:\Users\yshimane3\Documents\codes\dev-julia\TelescopeTasking.jl`


In [2]:
using JuMP, GLPK;

In [3]:
M = 1:3;
J = 1:15;
c = [12.7 22.5 8.9 20.8 13.6 12.4 24.8 19.1 11.5 17.4 24.7 6.8 21.7 14.3 10.5; 19.1 24.8 24.4 23.6 16.1 20.6 15.0 9.5 7.9 11.3 22.6 8.0 21.5 14.7 23.2;  18.6 14.1 22.7 9.9 24.2 24.5 20.8 12.9 17.7 11.9 18.7 10.1 9.1 8.9 7.7; 13.1 16.2 16.8 16.7 9.0 16.9 17.9 12.1 17.5 22.0 19.9 14.6 18.2 19.6 24.2];
w = [61 70 57 82 51 74 98 64 86 80 69 79 60 76 78; 50 57 61 83 81 79 63 99 82 59 83 91 59 99 91;91 81 66 63 59 81 87 90 65 55 57 68 92 91 86; 62 79 73 60 75 66 68 99 69 60 56 100 67 68 54];
Q = [1020 1460 1530];

In [4]:
model = Model(GLPK.Optimizer)
@variable(model, x[m in M, j in J], Bin);
@constraint(model, cov[j in J], sum(x[m, j] for m in M) >= 1);
@constraint(model, knp[m in M], sum(w[m, j] * x[m, j] for j in J) <= Q[m]);
@objective(model, Min, sum(c[m, j] * x[m, j] for m in M, j in J));

In [5]:
optimize!(model);
objective_value(model)

166.5

### Solving with column generation

This model has a block structure: each knapsack constraint defines an independent block and the set-partitioning constraints couple these independent blocks. By applying the Dantzig-Wolfe reformulation, each knapsack constraint forms a tractable subproblem and the set-partitioning constraints are handled in a master problem.

To write the model, you need JuMP and BlockDecomposition. The latter is an extension built on top of JuMP to model Dantzig-Wolfe and Benders decompositions. You will find more documentation about BlockDecomposition in the Decomposition & reformulation To optimize the problem, you need Coluna and a Julia package that provides a MIP solver such as GLPK.

Since we have already loaded JuMP and GLPK, we just need:

In [6]:
using BlockDecomposition, Coluna;

Next, you instantiate the solver and define the algorithm that you use to optimize the problem. In this case, the algorithm is a classic branch-and-price provided by Coluna.

In [7]:
coluna = optimizer_with_attributes(
    Coluna.Optimizer,
    "params" => Coluna.Params(
        solver = Coluna.Algorithm.TreeSearchAlgorithm() # default branch-cut-and-price
    ),
    "default_optimizer" => GLPK.Optimizer # GLPK for the master & the subproblems
)

MathOptInterface.OptimizerWithAttributes(Coluna.Optimizer, Pair{MathOptInterface.AbstractOptimizerAttribute, Any}[MathOptInterface.RawOptimizerAttribute("params") => Coluna.Params
  global_art_var_cost: Nothing nothing
  local_art_var_cost: Nothing nothing
  solver: Coluna.Algorithm.TreeSearchAlgorithm
, MathOptInterface.RawOptimizerAttribute("default_optimizer") => GLPK.Optimizer])

In `BlockDecomposition`, an axis is an index set of subproblems. Let `M_axis` be the index set of machines; it defines an axis along which we can implement the desired decomposition.

In [8]:
@axis(M_axis, M);

In this example, the axis `M_axis` defines one knapsack subproblem for each machine. For instance, the first machine index is 1 and is of type `BlockDecomposition.AxisId`:

In [9]:
M_axis[1]

typeof(M_axis[1])

BlockDecomposition.AxisId{:M_axis, Int64}

Jobs are not involved in the decomposition, set `J` of jobs thus stays as a classic range.

The model takes the form:

In [10]:
model = BlockModel(coluna)

A JuMP Model
├ solver: Coluna
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

You can write `BlockModel(coluna; direct_model = true)` to pass names of variables and constraints to Coluna.

In [11]:
@variable(model, x[m in M_axis, j in J], Bin);
@constraint(model, cov[j in J], sum(x[m, j] for m in M_axis) >= 1);
@constraint(model, knp[m in M_axis], sum(w[m, j] * x[m, j] for j in J) <= Q[m]);
@objective(model, Min, sum(c[m, j] * x[m, j] for m in M_axis, j in J));

This is the same model as above except that we use a `BlockModel` instead of a Model and `M_axis` as the set of machines instead of M. Therefore, `BlockDecomposition` will know which variables and constraints are involved in subproblems because one of their indices is a `BlockDecomposition.AxisId`.

You then apply a Dantzig-Wolfe decomposition along `M_axis`:

In [12]:
@dantzig_wolfe_decomposition(model, decomposition, M_axis)

where `decomposition` is a variable that contains information about the decomposition.

In [13]:
decomposition

Root - Annotation(BlockDecomposition.Master, BlockDecomposition.DantzigWolfe, lm = 1.0, um = 1.0, bp = 1.0, id = 2) with 3 subproblems :
	 2 => Annotation(BlockDecomposition.DwPricingSp, BlockDecomposition.DantzigWolfe, lm = 1.0, um = 1.0, bp = 1.0, id = 4) 
	 3 => Annotation(BlockDecomposition.DwPricingSp, BlockDecomposition.DantzigWolfe, lm = 1.0, um = 1.0, bp = 1.0, id = 5) 
	 1 => Annotation(BlockDecomposition.DwPricingSp, BlockDecomposition.DantzigWolfe, lm = 1.0, um = 1.0, bp = 1.0, id = 3) 


Once the decomposition is defined, you can retrieve the master and the subproblems to give additional information to the solver.

In [14]:
master = getmaster(decomposition)
subproblems = getsubproblems(decomposition)

3-element Vector{BlockDecomposition.SubproblemForm}:
 Subproblem formulation for M_axis = 1 contains :	 1.0 <= multiplicity <= 1.0

 Subproblem formulation for M_axis = 2 contains :	 1.0 <= multiplicity <= 1.0

 Subproblem formulation for M_axis = 3 contains :	 1.0 <= multiplicity <= 1.0


The multiplicity of a subproblem is the number of times that the same independent block shaped by the subproblem appears in the model. This multiplicity also specifies the number of solutions to the subproblem that can appear in the solution to the original problem.

In this GAP instance, the upper multiplicity is `1` because every subproblem is different, i.e., every machine is different and used at most once.

The lower multiplicity is `0` because a machine may stay unused. The multiplicity specifications take the form:

In [15]:
specify!.(subproblems, lower_multiplicity = 0, upper_multiplicity = 1)
getsubproblems(decomposition)

3-element Vector{BlockDecomposition.SubproblemForm}:
 Subproblem formulation for M_axis = 1 contains :	 0.0 <= multiplicity <= 1.0

 Subproblem formulation for M_axis = 2 contains :	 0.0 <= multiplicity <= 1.0

 Subproblem formulation for M_axis = 3 contains :	 0.0 <= multiplicity <= 1.0


The model is now fully defined. To solve it, you need to call:

In [16]:
optimize!(model)

Finally, you can retrieve the solution to the original formulation with JuMP methods. For example, if we want to know if job 3 is assigned to machine 1:

In [41]:
value(x[1,3])

1.0